In [1]:
from pathlib import Path
import os
from pathlib import Path
import sys
sys.path.append(os.path.abspath(".."))
from scripts.Explainer import Explainer
from scripts.Config_Kw_Dict import get_kw_dict
import torch
import numpy as np
import pickle
from pathlib import Path
from scripts.help_functions import Help_Functions
from scripts.recommender.recommenders_architecture import MLP, VAE
from scripts.Config_Kw_Dict import get_kw_dict
from scripts.Evaluation import Evaluation



In [2]:
data_name = "ML1M" # Can be ML1M, Yahoo, Pinterest
recommender_name = "MLP" # Can be MLP, VAE
kw_dict=get_kw_dict()



In [3]:
DP_DIR = Path("scripts", 'checkpoints') 
export_dir = Path(os.getcwd())
files_path = Path(export_dir, DP_DIR)
## path for loading LXR explainer
lxr_path=f'LXR-rep_{data_name}_{recommender_name}_{49}.pt'
## path for loading the records
record_path=f'Records_LF_{data_name}_{recommender_name}.pkl' 

In [4]:
export_dir

PosixPath('/Users/meva4969/VS-Code/lxr_repro_2025')

## loading data

In [5]:
with open(Path(files_path,record_path), 'rb') as file:
    record_LF = pickle.load(file)

## loading recommender

In [7]:
def load_recommender():
        #data_name=self.data_name
        if recommender_name=='MLP':

            recommender = MLP(data_name, **kw_dict)
        elif recommender_name=='VAE':
            recommender = VAE(data_name, **kw_dict)
        recommender_checkpoint = torch.load(Path(files_path, kw_dict['recommender_path'][(data_name, recommender_name)] ), map_location=kw_dict['device'])
        recommender.load_state_dict(recommender_checkpoint)
        recommender.eval()
        for param in recommender.parameters():
            param.requires_grad= False
        return recommender
recommender = load_recommender()

## Loading LXR explainer

In [8]:
exp_hid_szie=kw_dict['Predefined_hyperparameters'][recommender_name][data_name]['explainer_hidden_size']
num_items=kw_dict['num_items'][data_name]


def load_explainer():
    explainer = Explainer(num_items, num_items, exp_hid_szie)
    lxr_checkpoint = torch.load(Path(files_path, lxr_path), map_location=torch.device("mps" if torch.backends.mps.is_available() else "cpu"))

    explainer.load_state_dict(lxr_checkpoint)
    explainer.eval()
    for param in explainer.parameters():
        param.requires_grad= False
    return explainer

explainer = load_explainer()

## Getting LXR outputs for the same users and targ items in LF explainer

In [9]:
MPRR_R, Mask=[],[]
coverage=[]

for j in range(len(record_LF)):

    user_id = record_LF[j]['user_id']
    user_tensor = record_LF[j]['user_tensor'].to(kw_dict['device'])
    ## Target item for testing dataset
    i1 = record_LF[j]['targ_item']
    i1_index=record_LF[j]['targ_index']
    i1_vector = record_LF[j]['items_array']
    i1_tensor = torch.Tensor(i1_vector).to(kw_dict['device'])
    evaluation=Evaluation(data_name,recommender_name,explainer,recommender, kw_dict, k=10 )
    p,q= evaluation(user_tensor, i1,i1_index, i1_tensor)

    if p is not None:
        MPRR_R.append(p)
        Mask.append(q)

coverage.append(len(MPRR_R))  



RuntimeError: Tensor for argument weight is on cpu but expected on mps

## Rank Bias overlaped

In [ ]:
RBO_list=[]
Mask_LF=record_LF['mask']
Mask_LXR=record_LXR['mask']

for i in range(len(Mask)):

    LXR_Mask=[k for k , v in Mask_LXR[i][0:record_LXR['MPNR'][i]]]
    LF_Mask=[k for k , v in Mask_LF[i][0:record_LF['MPNR'][i]]]

    LF_Mask = [int(t.to("cpu")) for t in LF_Mask]
    RBO_list.append(rbo.RankingSimilarity(LXR_Mask, LF_Mask).rbo())

In [ ]:
import os
print(os.getcwd())
